# Final Project 2 model

Application-only neural network adapted from Dilraj’s recipe and evaluated with Gil’s protocol. Gender is excluded; age is retained with documented fairness tradeoffs. See the accompanying Word document for the team contributions and decisions.

**Run this notebook first**, then `Final_Error_Analysis.ipynb`. Place `application_train.csv` beside the notebooks. The first run trains the fixed recipe on CPU and writes intermediate results into the hidden local `.project2_runtime` folder; later runs reuse that checkpoint. No supporting repository files are required. The optional nonrandom-split rerun also needs `previous_application.csv`.

The submitted results use seed 42 and a 0.167 cutoff selected using blackbox F1. Training and saved predictions were reproduced exactly in the tested environment. A different software environment may vary; consistency checks stop before overwriting the submitted error review if predictions differ.

Tested packages: numpy 2.5.3, pandas 3.0.6, scipy 1.18.1, scikit-learn 1.9.1, torch 2.14.1, matplotlib 3.11.2, joblib 1.6.0, ipython 9.17.1. Install these into your notebook kernel if needed.

## Imports and fixed settings

In [1]:
"""Final application-only neural network; adapted from Dilraj's nn1 and Gil's audit.
Gender excluded. All fitted transformations use training rows only.
"""
import copy,gc,hashlib,json,math,random,time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import joblib
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import MissingIndicator,SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,confusion_matrix,average_precision_score,brier_score_loss,log_loss
RANDOM_STATE=42
DEVICE=torch.device('cpu')
MAX_EPOCHS=40
PATIENCE=7
BATCH_SIZE=2048
CONFIG=dict(name='final_application',feature_set='application_only',hidden=[128,64],dropout=.35,input_dropout=.08,weight_decay=.03)

from IPython.display import display
RUNTIME=Path('.project2_runtime')
(RUNTIME/'selected').mkdir(parents=True,exist_ok=True)
FORCE_RETRAIN=False


## Training-only preprocessing and feature engineering

In [2]:
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


In [3]:
def safe_divide(numerator, denominator):
    return numerator.div(denominator.replace(0, np.nan))


In [4]:
def engineer_features(raw, engineered=True):
    x = raw.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore').copy()
    employment_sentinel = x['DAYS_EMPLOYED'].eq(365243)
    x['DAYS_EMPLOYED'] = x['DAYS_EMPLOYED'].mask(employment_sentinel)
    x = x.drop(columns=['CODE_GENDER'], errors='ignore')
    if not engineered:
        return x.replace([np.inf, -np.inf], np.nan)
    extra = {}
    extra['EMPLOYMENT_UNKNOWN'] = employment_sentinel.astype(float)
    extra['APPLICATION_MISSING_COUNT'] = x.isna().sum(axis=1)
    extra['AGE_YEARS'] = -x['DAYS_BIRTH'] / 365.25
    extra['EMPLOYMENT_YEARS'] = -x['DAYS_EMPLOYED'] / 365.25
    ratios = {'CREDIT_TO_INCOME': ('AMT_CREDIT', 'AMT_INCOME_TOTAL'), 'ANNUITY_TO_INCOME': ('AMT_ANNUITY', 'AMT_INCOME_TOTAL'), 'CREDIT_TO_ANNUITY': ('AMT_CREDIT', 'AMT_ANNUITY'), 'ANNUITY_TO_CREDIT': ('AMT_ANNUITY', 'AMT_CREDIT'), 'GOODS_TO_CREDIT': ('AMT_GOODS_PRICE', 'AMT_CREDIT'), 'GOODS_TO_INCOME': ('AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL'), 'INCOME_PER_PERSON': ('AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS'), 'CREDIT_PER_PERSON': ('AMT_CREDIT', 'CNT_FAM_MEMBERS'), 'ANNUITY_PER_PERSON': ('AMT_ANNUITY', 'CNT_FAM_MEMBERS'), 'CHILDREN_FRACTION': ('CNT_CHILDREN', 'CNT_FAM_MEMBERS'), 'EMPLOYMENT_TO_AGE': ('DAYS_EMPLOYED', 'DAYS_BIRTH'), 'REGISTRATION_TO_AGE': ('DAYS_REGISTRATION', 'DAYS_BIRTH'), 'ID_CHANGE_TO_AGE': ('DAYS_ID_PUBLISH', 'DAYS_BIRTH'), 'PHONE_CHANGE_TO_AGE': ('DAYS_LAST_PHONE_CHANGE', 'DAYS_BIRTH'), 'SOCIAL_DEFAULT_RATE_30': ('DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_30_CNT_SOCIAL_CIRCLE'), 'SOCIAL_DEFAULT_RATE_60': ('DEF_60_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE')}
    for name, (numerator, denominator) in ratios.items():
        extra[name] = safe_divide(x[numerator], x[denominator])
    extra['CREDIT_MINUS_GOODS'] = x['AMT_CREDIT'] - x['AMT_GOODS_PRICE']
    extra['CREDIT_MINUS_GOODS_FRACTION'] = safe_divide(extra['CREDIT_MINUS_GOODS'], x['AMT_CREDIT'])
    extra['INCOME_MINUS_ANNUITY'] = x['AMT_INCOME_TOTAL'] - x['AMT_ANNUITY']
    extra['ADULT_COUNT'] = (x['CNT_FAM_MEMBERS'] - x['CNT_CHILDREN']).clip(lower=1)
    extra['INCOME_PER_ADULT'] = safe_divide(x['AMT_INCOME_TOTAL'], extra['ADULT_COUNT'])
    extra['CAR_AGE_TO_AGE'] = safe_divide(x['OWN_CAR_AGE'], extra['AGE_YEARS'])
    extra['EMPLOYMENT_START_AGE'] = extra['AGE_YEARS'] - extra['EMPLOYMENT_YEARS']
    ext = x[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']]
    extra['EXT_MEAN'] = ext.mean(axis=1)
    extra['EXT_STD'] = ext.std(axis=1)
    extra['EXT_MIN'] = ext.min(axis=1)
    extra['EXT_MAX'] = ext.max(axis=1)
    extra['EXT_RANGE'] = extra['EXT_MAX'] - extra['EXT_MIN']
    extra['EXT_MISSING_COUNT'] = ext.isna().sum(axis=1)
    extra['EXT_PRODUCT'] = ext.prod(axis=1, min_count=3)
    for a, b in [(1, 2), (1, 3), (2, 3)]:
        extra[f'EXT_{a}_TIMES_{b}'] = x[f'EXT_SOURCE_{a}'] * x[f'EXT_SOURCE_{b}']
    extra['EXT_MEAN_TIMES_AGE'] = extra['EXT_MEAN'] * extra['AGE_YEARS']
    extra['EXT_MEAN_TO_CREDIT_INCOME'] = safe_divide(extra['EXT_MEAN'], extra['CREDIT_TO_INCOME'])
    extra['EXT_MEAN_TIMES_ANNUITY_CREDIT'] = extra['EXT_MEAN'] * extra['ANNUITY_TO_CREDIT']
    documents = [c for c in x if c.startswith('FLAG_DOCUMENT_')]
    bureau = [c for c in x if c.startswith('AMT_REQ_CREDIT_BUREAU_')]
    housing = [c for c in x if c.endswith(('_AVG', '_MODE', '_MEDI')) and pd.api.types.is_numeric_dtype(x[c])]
    contacts = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']
    addresses = [c for c in x if '_NOT_' in c]
    extra['DOCUMENT_COUNT'] = x[documents].sum(axis=1)
    extra['CONTACT_COUNT'] = x[contacts].sum(axis=1)
    extra['ADDRESS_MISMATCH_COUNT'] = x[addresses].sum(axis=1)
    extra['HOUSING_MISSING_COUNT'] = x[housing].isna().sum(axis=1)
    extra['BUREAU_ACTIVITY_INDEX'] = x[bureau].sum(axis=1, min_count=1)
    extra['BUREAU_MISSING_COUNT'] = x[bureau].isna().sum(axis=1)
    extra['APPLICATION_HOUR_SIN'] = np.sin(2 * np.pi * x['HOUR_APPR_PROCESS_START'] / 24)
    extra['APPLICATION_HOUR_COS'] = np.cos(2 * np.pi * x['HOUR_APPR_PROCESS_START'] / 24)
    return pd.concat([x, pd.DataFrame(extra, index=x.index)], axis=1).replace([np.inf, -np.inf], np.nan)


In [5]:
def fit_preprocessing(train_frame, validation_frame):
    numeric = train_frame.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical = train_frame.select_dtypes(include=['object', 'category']).columns.tolist()
    continuous = [c for c in numeric if train_frame[c].nunique() > 2]
    binary = [c for c in numeric if c not in continuous]
    continuous_pipe = Pipeline([('impute', SimpleImputer(strategy='median', keep_empty_features=True)), ('quantile', QuantileTransformer(n_quantiles=512, output_distribution='normal', subsample=100000, random_state=RANDOM_STATE))])
    category_pipe = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Missing')), ('encode', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=False, dtype=np.float32))])
    prep = ColumnTransformer([('continuous', continuous_pipe, continuous), ('binary', SimpleImputer(strategy='most_frequent', keep_empty_features=True), binary), ('missing', MissingIndicator(features='all', error_on_new=False), numeric), ('categorical', category_pipe, categorical)], sparse_threshold=0, n_jobs=1)
    a = np.clip(prep.fit_transform(train_frame), -4, 4).astype(np.float32)
    b = np.clip(prep.transform(validation_frame), -4, 4).astype(np.float32)
    names = prep.get_feature_names_out()
    variance = VarianceThreshold(threshold=0.0)
    a = variance.fit_transform(a)
    b = variance.transform(b)
    names = names[variance.get_support()]
    import hashlib
    groups, unique = ({}, [])
    for j in range(a.shape[1]):
        key = hashlib.sha256(a[:, j].tobytes()).digest()
        prior = groups.setdefault(key, [])
        if not any((np.array_equal(a[:, j], a[:, k]) for k in prior)):
            unique.append(j)
            prior.append(j)
    unique = np.asarray(unique)
    bundle = {'preprocessor': prep, 'variance': variance, 'unique': unique, 'feature_names': names[unique]}
    a, b = (np.ascontiguousarray(a[:, unique]), np.ascontiguousarray(b[:, unique]))
    assert np.isfinite(a).all() and np.isfinite(b).all()
    return (bundle, a, b)


## Neural network

In [6]:
class CreditNet(nn.Module):

    def __init__(self, input_dim, hidden=(128, 64), dropout=0.2, input_dropout=0.03):
        super().__init__()
        self.input_dropout = nn.Dropout(input_dropout)
        layers, width = ([], input_dim)
        for next_width in hidden:
            layers.extend([nn.Linear(width, next_width), nn.BatchNorm1d(next_width), nn.SiLU(), nn.Dropout(dropout)])
            width = next_width
        layers.append(nn.Linear(width, 1))
        self.deep = nn.Sequential(*layers)
        self.linear = nn.Linear(input_dim, 1)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return (self.deep(self.input_dropout(x)) + self.linear(x)).squeeze(1)


In [7]:
def make_model(config, input_dim):
    return CreditNet(input_dim, tuple(config['hidden']), config['dropout'], config.get('input_dropout', 0.03)).to(DEVICE)


In [8]:
@torch.no_grad()
def predict_network(model, x, batch_size=8192):
    model.eval()
    values = []
    for start in range(0, len(x), batch_size):
        inputs = torch.as_tensor(x[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        values.append(torch.sigmoid(model(inputs)).cpu().numpy())
    return np.concatenate(values)


## Fit the model and restore the best dev checkpoint

In [9]:
def train_trial(config, x_train, x_validation, seed=42):
    seed_everything(seed)
    start = time.perf_counter()
    model = make_model(config, x_train.shape[1])
    prior = float(y_train.mean())
    nn.init.constant_(model.linear.bias, math.log(prior / (1 - prior)))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.get('lr', 0.001), weight_decay=config.get('weight_decay', 0.01))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, min_lr=2e-05)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(config.get('pos_weight', 1.0), device=DEVICE))
    xt = torch.as_tensor(np.ascontiguousarray(x_train), dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(y_train, dtype=torch.float32, device=DEVICE)
    best_auc, best_epoch, best_state, stale = (-np.inf, 0, None, 0)
    history = []
    for epoch in range(1, config.get('max_epochs', MAX_EPOCHS) + 1):
        model.train()
        order = torch.randperm(len(xt), device=DEVICE)
        loss_sum = 0.0
        for indices in order.split(BATCH_SIZE):
            if len(indices) < 2:
                continue
            xb, yb = (xt[indices], yt[indices])
            noise = config.get('noise', 0.02)
            if noise:
                xb = xb + noise * torch.randn_like(xb)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            loss_sum += float(loss.detach()) * len(indices)
        pred = predict_network(model, x_validation)
        auc = roc_auc_score(y_val, pred)
        history.append({'epoch': epoch, 'train_loss': loss_sum / len(xt), 'val_auc': auc, 'lr': optimizer.param_groups[0]['lr']})
        scheduler.step(auc)
        material_improvement = auc > best_auc + 1e-05
        if auc > best_auc:
            best_auc, best_epoch = (auc, epoch)
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        stale = 0 if material_improvement else stale + 1
        if epoch == 1 or epoch % 5 == 0:
            print(f"{config['name']} seed={seed} epoch={epoch:02d}: val AUC={auc:.6f}, best={best_auc:.6f}", flush=True)
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    val_pred = predict_network(model, x_validation)
    elapsed = time.perf_counter() - start
    checkpoint_name = f"{config['name']}_seed{seed}.pt"
    torch.save({'state_dict': best_state, 'config': config, 'seed': seed, 'input_dim': x_train.shape[1], 'best_epoch': best_epoch}, ARTIFACT_DIR / checkpoint_name)
    record = {'name': config['name'], 'feature_set': config['feature_set'], 'seed': seed, 'val_auc': float(roc_auc_score(y_val, val_pred)), 'best_epoch': best_epoch, 'epochs_run': epoch, 'seconds': round(elapsed, 1), 'checkpoint': checkpoint_name, 'config': config.copy(), 'history': history, 'val_pred': val_pred}
    print(f"Finished {config['name']}: AUC={record['val_auc']:.6f}, epoch={best_epoch}, {elapsed:.1f}s", flush=True)
    del model, xt, yt, optimizer
    gc.collect()
    return record


## Group metrics and uncertainty

In [10]:
def fairness_tables(predictions, attributes, cutoff):
    """Audit TARGET=1 difficulty; undefined rates stay NaN, never zero."""
    data = predictions.merge(attributes[['SK_ID_CURR', 'CODE_GENDER', 'DAYS_BIRTH']],
                             on='SK_ID_CURR', validate='one_to_one')
    data['age_group'] = pd.cut(-data.DAYS_BIRTH / 365.25, [0, 30, 45, 60, np.inf],
        right=False, labels=['Under 30', '30 to under 45', '45 to under 60', '60 and older'])
    data['label'] = data.predicted_probability.ge(cutoff)
    rows = []
    for attribute in ['CODE_GENDER', 'age_group']:
        for group, part in data.groupby(attribute, observed=True, dropna=False):
            actual = part.actual_TARGET.eq(1); label = part.label
            tp=int((actual & label).sum()); fn=int((actual & ~label).sum())
            fp=int((~actual & label).sum()); tn=int((~actual & ~label).sum())
            counts = {'false_positive_rate': (fp, fp+tn), 'recall': (tp, tp+fn),
                      'precision': (tp, tp+fp), 'npv': (tn, tn+fn),
                      'specificity': (tn, tn+fp), 'false_negative_rate': (fn, fn+tp),
                      'favorable_prediction_rate': (tn+fn, len(part)),
                      'difficulty_prevalence': (tp+fn, len(part))}
            row=dict(attribute=attribute, group=str(group), n=len(part),
                     positives=tp+fn, negatives=tn+fp, tp=tp, fn=fn, fp=fp, tn=tn)
            for metric, (success, total) in counts.items():
                rate=success/total if total else np.nan
                row[metric]=rate; row[metric+'_denominator']=total
                # Descriptive 95% Wilson interval; does not account for repeated dev selection.
                if total:
                    z=1.959963984540054; den=1+z*z/total
                    center=(rate+z*z/(2*total))/den
                    half=z*np.sqrt(rate*(1-rate)/total+z*z/(4*total*total))/den
                    row[metric+'_lower']=max(0.,center-half); row[metric+'_upper']=min(1.,center+half)
                else:
                    row[metric+'_lower']=np.nan; row[metric+'_upper']=np.nan
            row['small_group_warning']=min(tp+fn,tn+fp,tp+fp,tn+fn)<30
            rows.append(row)
    groups=pd.DataFrame(rows)
    gaps=[]
    for attribute in groups.attribute.unique():
        part=groups[groups.attribute.eq(attribute)]
        for metric in ['false_positive_rate','recall','precision','npv','favorable_prediction_rate','specificity']:
            eligible=part[part[metric+'_denominator'].ge(30)&part[metric].notna()]
            if len(eligible)>=2:
                lo=eligible.loc[eligible[metric].idxmin()]; hi=eligible.loc[eligible[metric].idxmax()]
                gaps.append(dict(attribute=attribute,metric=metric,eligible_groups=len(eligible),
                    excluded_groups='; '.join(part.loc[~part.index.isin(eligible.index),'group']),
                    lowest_group=lo['group'],highest_group=hi['group'],
                    minimum=lo[metric],maximum=hi[metric],gap_pp=100*(hi[metric]-lo[metric])))
    return groups, pd.DataFrame(gaps)


## Data partitions and evaluation

In [11]:
def load_data(path='application_train.csv'):
    df=pd.read_csv(path)
    assert df.SK_ID_CURR.is_unique and set(df.TARGET.unique())=={0,1}
    return df


In [12]:
def main_partitions(df):
    train,test=train_test_split(df.index,test_size=.20,random_state=42,stratify=df.TARGET)
    train,dev=train_test_split(train,test_size=.25,random_state=42,stratify=df.loc[train,'TARGET'])
    eye_ids=set(json.loads(Path('.project2_runtime/selected/dev_split_manifest.json').read_text())['eyeball_ids'])
    eye=dev[df.loc[dev,'SK_ID_CURR'].isin(eye_ids).to_numpy()]
    black=dev[~df.loc[dev,'SK_ID_CURR'].isin(eye_ids).to_numpy()]
    parts={'train':train,'blackbox':black,'eyeball':eye,'test':test}
    assert [len(x) for x in parts.values()]==[184506,61002,500,61503]
    assert set(df.loc[eye,'SK_ID_CURR'])==eye_ids
    assert sum(map(len,parts.values()))==len(set().union(*map(set,parts.values())))==len(df)
    return parts


In [13]:
def transform(bundle,raw):
    frame=engineer_features(raw)
    assert 'CODE_GENDER' not in frame and 'TARGET' not in frame and 'SK_ID_CURR' not in frame
    a=np.clip(bundle['preprocessor'].transform(frame),-4,4).astype(np.float32)
    a=bundle['variance'].transform(a)[:,bundle['unique']]
    assert np.isfinite(a).all()
    return np.ascontiguousarray(a)


In [14]:
def select_threshold(y,p):
    thresholds=np.arange(10,501)/1000;order=np.argsort(p);pp=p[order];yy=np.asarray(y)[order]
    tp=np.r_[np.cumsum(yy[::-1])[::-1],0];at=np.searchsorted(pp,thresholds)
    score=2*tp[at]/(len(pp)-at+yy.sum())
    return float(thresholds[np.argmax(score)])


In [15]:
def metrics(y,p,threshold):
    tn,fp,fn,tp=confusion_matrix(y,p>=threshold,labels=[0,1]).ravel()
    return dict(n=len(y),prevalence=float(np.mean(y)),auc=roc_auc_score(y,p),average_precision=average_precision_score(y,p),accuracy=(tp+tn)/len(y),precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),false_positive_rate=fp/(fp+tn),brier=brier_score_loss(y,p),log_loss=log_loss(y,p),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))


In [16]:
def fit_recipe(df,train,blackbox,folder,seed=42):
    global y_train,y_val,ARTIFACT_DIR
    ARTIFACT_DIR=Path(folder);ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
    y_train=df.loc[train,'TARGET'].to_numpy(dtype=np.float32);y_val=df.loc[blackbox,'TARGET'].to_numpy()
    bundle,a,b=fit_preprocessing(engineer_features(df.loc[train]),engineer_features(df.loc[blackbox]))
    record=train_trial(CONFIG,a,b,seed=seed);prob=record.pop('val_pred')
    joblib.dump(bundle,ARTIFACT_DIR/'preprocessor.joblib')
    (ARTIFACT_DIR/'training_record.json').write_text(json.dumps(record,indent=2))
    return bundle,ARTIFACT_DIR/record['checkpoint'],select_threshold(y_val,prob)


In [17]:
def evaluate_frozen(df,parts,folder,checkpoint_path,threshold):
    folder=Path(folder);bundle=joblib.load(folder/'preprocessor.joblib')
    ck=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    model=make_model(ck['config'],ck['input_dim']);model.load_state_dict(ck['state_dict'])
    rows=[]
    for partition,idx in parts.items():
        p=predict_network(model,transform(bundle,df.loc[idx]));y=df.loc[idx,'TARGET'].to_numpy()
        pred=pd.DataFrame({'SK_ID_CURR':df.loc[idx,'SK_ID_CURR'].to_numpy(),'actual_TARGET':y,'predicted_probability':p,'predicted_label':(p>=threshold).astype(int)})
        if partition!='train':
            pred.to_json(folder/(partition+'_predictions.json'),orient='records',indent=2)
            groups,gaps=fairness_tables(pred,df,threshold)
            groups.to_json(folder/(partition+'_fairness.json'),orient='records',indent=2)
            gaps.to_json(folder/(partition+'_fairness_gaps.json'),orient='records',indent=2)
        rows.append(dict(partition=partition,threshold=threshold,selected_epoch=ck['best_epoch'],**metrics(y,p,threshold)))
    result=pd.DataFrame(rows);result.to_json(folder/'metrics.json',orient='records',indent=2)
    meta=dict(model_id='final_application_peer_seed42',threshold=threshold,seed=42,selected_epoch=ck['best_epoch'],config=ck['config'],input_dim=ck['input_dim'],excluded_predictors=['CODE_GENDER'],age_retained=True,selection_partition='blackbox',historical_exposure=True)
    (folder/'model_info.json').write_text(json.dumps(meta,indent=2))
    return result


## Fixed eyeball pool and selection evidence
The pool was set before this final comparison. Reusing historical dev data does not create a fresh holdout. The following small tables preserve the completed comparison, not a new search.

In [18]:
manifest={'version': 1, 'seed': 42, 'eyeball_ids': [102331, 102870, 102899, 102959, 103495, 103949, 104414, 104571, 104589, 104795, 105679, 109338, 109499, 113347, 114874, 115444, 116561, 116881, 117001, 117306, 118432, 118580, 119024, 119539, 119584, 121564, 121571, 121746, 122835, 122888, 123821, 124273, 125631, 126107, 126463, 127076, 127443, 128852, 130591, 131301, 131596, 132196, 133395, 134177, 135852, 135957, 136030, 136969, 137209, 137409, 137943, 141446, 141448, 142712, 142924, 143107, 143406, 145132, 145299, 145324, 145505, 146432, 147379, 147583, 148004, 148845, 148988, 149036, 150403, 150523, 151140, 151473, 153587, 153880, 153900, 154709, 154793, 154799, 157166, 157274, 157369, 158364, 160093, 160288, 160460, 160558, 160720, 161216, 161245, 161870, 162434, 162601, 162699, 162984, 163455, 163613, 165404, 165609, 165927, 166992, 167688, 168007, 168141, 168501, 169231, 169791, 170085, 170405, 170704, 170849, 170889, 170916, 171000, 174273, 175134, 175159, 175366, 176109, 178102, 179410, 180235, 181636, 181841, 182100, 182729, 183333, 185182, 185811, 186114, 186940, 187770, 190949, 191652, 191856, 192392, 192556, 193187, 194741, 196300, 196458, 197110, 197272, 197825, 198515, 198729, 198750, 199463, 199721, 201086, 202332, 203184, 203364, 203891, 205673, 206191, 206245, 207579, 207771, 207833, 208692, 209413, 209864, 210029, 210120, 210476, 210972, 211179, 212214, 214568, 214618, 215363, 216244, 216504, 217456, 218447, 218621, 218892, 218936, 219386, 220174, 220267, 221130, 221539, 221841, 222610, 223818, 223857, 224555, 224913, 224970, 226155, 226202, 227682, 228526, 228527, 229009, 229424, 229524, 229822, 230550, 232734, 232769, 234565, 235628, 237610, 237737, 238008, 238672, 238872, 239166, 239234, 241611, 241691, 243113, 243384, 243508, 244663, 244927, 245946, 248116, 250190, 251126, 251842, 252091, 252105, 252281, 252516, 253053, 254373, 256025, 256668, 257587, 259646, 259778, 260167, 262611, 262785, 266921, 266980, 267003, 267850, 269861, 270164, 270314, 270525, 272552, 272655, 272713, 276065, 276506, 276559, 277868, 277895, 279697, 280425, 280747, 281139, 281316, 281976, 283428, 283692, 283710, 284268, 284869, 286037, 286919, 289394, 289468, 290629, 290914, 291286, 292932, 294096, 294778, 294843, 294846, 294850, 296131, 296899, 300441, 300817, 300846, 301663, 301677, 304239, 304988, 305001, 305291, 305929, 306053, 310451, 310572, 310675, 311104, 311955, 312083, 312784, 312850, 313484, 313498, 315921, 316362, 316439, 317744, 318120, 318900, 319711, 321001, 322259, 322871, 323927, 324682, 325186, 326648, 327428, 327529, 328927, 329809, 331040, 331168, 331230, 331715, 332030, 332654, 333437, 333920, 333939, 334012, 334569, 334647, 337084, 339836, 339900, 340269, 340398, 341851, 343356, 344148, 344372, 345488, 345529, 347367, 347957, 347973, 348218, 348790, 349405, 349772, 350038, 351931, 352462, 352857, 356486, 358003, 359056, 359853, 359968, 360057, 360193, 361395, 362455, 362566, 363275, 364581, 364641, 365163, 365730, 366157, 367082, 368217, 369626, 371108, 371466, 371888, 372873, 373343, 375430, 375981, 377601, 378571, 379165, 379267, 379917, 380299, 380721, 380740, 381070, 381634, 383371, 384793, 384840, 386204, 386612, 386962, 387430, 388509, 389560, 389684, 389918, 390517, 391470, 391816, 392273, 394026, 394229, 394482, 394597, 395390, 396781, 398235, 398826, 400455, 400464, 402176, 403057, 403184, 403870, 404347, 404422, 404688, 405549, 405943, 406030, 406180, 406430, 407080, 407292, 407438, 407637, 407690, 408104, 408812, 410527, 412000, 412964, 413366, 415676, 415912, 415961, 417805, 419288, 419607, 420315, 421485, 421954, 422427, 424085, 424549, 424555, 425280, 425677, 425727, 426809, 427857, 427954, 428528, 430392, 430506, 432196, 433923, 434906, 435516, 435930, 436136, 436301, 436317, 437812, 438218, 439974, 440797, 441033, 441791, 442471, 442763, 443654, 443889, 444036, 444534, 444619, 445069, 445150, 445426, 445443, 447281, 448228, 449340, 449617, 449715, 450157, 450514, 451704, 452570, 453163, 454072, 454452, 454760, 454906, 455202, 455516, 455688], 'eyeball_count': 500, 'blackbox_count': 61002, 'note': 'Fixed existing eyeball membership. Earlier experiments used full dev; this cannot undo historical exposure.'}
(RUNTIME/'selected/dev_split_manifest.json').write_text(json.dumps(manifest,indent=2))
selection={'date': '2026-10-14', 'selected': 'peer_recipe', 'seed': 42, 'threshold': 0.167, 'epoch': 23, 'selection_metric': 'blackbox AUC', 'threshold_metric': 'blackbox F1', 'mean_auc_gain': 0.0078864204937935, 'seed_wins': 3, 'decision': 'Adopt for course project within application-only gender-excluded scope. Better ranking/F1; accept documented recall and disparity tradeoffs without claiming fairness approval.', 'fairness': 'No quantitative acceptance tolerance agreed. Report both own-F1 and matched-FPR disparities; not a deployment certification.', 'test_or_eyeball_used_for_this_selection': False, 'historical_exposure': True}
peer_comparison=[{'name': 'baseline', 'auc': 0.7470749426089768, 'f1': 0.29647252761932535, 'recall': 0.4407445008460236, 'precision': 0.22337929461118966, 'fpr': 0.13458280578490292, 'gender_gap_pp': 5.6704903194953715, 'age_gap_pp': 22.307376239469818, 'auc_gain': 0, 'seed_wins': 0}, {'name': 'quantile_current', 'auc': 0.745818221241418, 'f1': 0.29167794976997125, 'recall': 0.4115736040609137, 'precision': 0.22616098222748107, 'fpr': 0.12390106460759313, 'gender_gap_pp': 5.70870178637631, 'age_gap_pp': 21.926937474893677, 'auc_gain': -0.0012567213675587958, 'seed_wins': 0}, {'name': 'peer_recipe', 'auc': 0.7549613631027702, 'f1': 0.3023544972797112, 'recall': 0.4116412859560068, 'precision': 0.2393723551643204, 'fpr': 0.11519280037567392, 'gender_gap_pp': 5.9099037681350195, 'age_gap_pp': 20.63910601059432, 'auc_gain': 0.007886420493793511, 'seed_wins': 3}]
display(pd.DataFrame(peer_comparison))
print(json.dumps(selection,indent=2))


{
  "date": "2026-10-14",
  "selected": "peer_recipe",
  "seed": 42,
  "threshold": 0.167,
  "epoch": 23,
  "selection_metric": "blackbox AUC",
  "threshold_metric": "blackbox F1",
  "mean_auc_gain": 0.0078864204937935,
  "seed_wins": 3,
  "decision": "Adopt for course project within application-only gender-excluded scope. Better ranking/F1; accept documented recall and disparity tradeoffs without claiming fairness approval.",
  "fairness": "No quantitative acceptance tolerance agreed. Report both own-F1 and matched-FPR disparities; not a deployment certification.",
  "test_or_eyeball_used_for_this_selection": false,
  "historical_exposure": true
}


,name,auc,f1,recall,precision,fpr,gender_gap_pp,age_gap_pp,auc_gain,seed_wins
0,baseline,0.747075,0.296473,0.440745,0.223379,0.134583,5.670490,22.307376,0.000000,0
1,quantile_current,0.745818,0.291678,0.411574,0.226161,0.123901,5.708702,21.926937,-0.001257,0
2,peer_recipe,0.754961,0.302354,0.411641,0.239372,0.115193,5.909904,20.639106,0.007886,3


## Train or load the fixed main model
The dev-selected threshold must reproduce 0.167 before this run is treated as the submitted final model. Test and eyeball results do not select it.

In [19]:
torch.set_num_threads(1)
df=load_data()
parts=main_partitions(df)
folder=RUNTIME/'selected'
checkpoint=folder/'final_application_seed42.pt'
if FORCE_RETRAIN or not checkpoint.exists():
    _,checkpoint,selected_threshold=fit_recipe(df,parts['train'],parts['blackbox'],folder)
    assert selected_threshold == .167, 'Different dev cutoff: review this rerun before replacing final outputs.'
results=evaluate_frozen(df,parts,folder,checkpoint,.167)
display(results.round(5))


final_application seed=42 epoch=01: val AUC=0.745532, best=0.745532
final_application seed=42 epoch=05: val AUC=0.750470, best=0.750470
final_application seed=42 epoch=10: val AUC=0.751206, best=0.753020
final_application seed=42 epoch=15: val AUC=0.754338, best=0.754338
final_application seed=42 epoch=20: val AUC=0.754320, best=0.754637
final_application seed=42 epoch=25: val AUC=0.754265, best=0.754731
final_application seed=42 epoch=30: val AUC=0.754530, best=0.754731
Finished final_application: AUC=0.754731, epoch=23, 27.7s


<ipython-input-5-023603609961>:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = train_frame.select_dtypes(include=['object', 'category']).columns.tolist()


,partition,threshold,selected_epoch,n,prevalence,auc,average_precision,accuracy,precision,recall,f1,false_positive_rate,brier,log_loss,tn,fp,fn,tp
0,train,0.167,23,184506,0.08073,0.77635,0.26451,0.85919,0.26434,0.41739,0.32368,0.10201,0.06673,0.24065,152309,17302,8678,6217
1,blackbox,0.167,23,61002,0.08074,0.75473,0.23643,0.85255,0.24473,0.39614,0.30255,0.10737,0.06819,0.24740,50056,6021,2974,1951
2,eyeball,0.167,23,500,0.08000,0.68652,0.15903,0.83800,0.18462,0.30000,0.22857,0.11522,0.07254,0.26576,407,53,28,12
3,test,0.167,23,61503,0.08073,0.75668,0.24384,0.85511,0.24975,0.39658,0.30648,0.10462,0.06785,0.24639,50623,5915,2996,1969


## Fairness at the selected cutoff
Removing gender does not guarantee parity. Report FPR and recall together, with denominators and small-group warnings.

In [20]:
fairness=pd.read_json(folder/'blackbox_fairness.json')
display(fairness[['attribute','group','n','false_positive_rate','recall','precision','npv','favorable_prediction_rate','small_group_warning']].round(4))
display(pd.read_json(folder/'blackbox_fairness_gaps.json').round(3))


,attribute,group,n,false_positive_rate,recall,precision,npv,favorable_prediction_rate,small_group_warning
0,CODE_GENDER,F,40195,0.0890,0.3567,0.2323,0.9494,0.8923,False
1,CODE_GENDER,M,20806,0.1441,0.4489,0.2596,0.9324,0.8251,False
2,CODE_GENDER,XNA,1,1.0000,NaN,0.0000,NaN,0.0000,True
3,age_group,Under 30,8900,0.2202,0.5511,0.2420,0.9316,0.7424,False
4,age_group,30 to under 45,24675,0.1235,0.4230,0.2554,0.9382,0.8492,False
5,age_group,45 to under 60,20531,0.0706,0.3037,0.2311,0.9503,0.9142,False
6,age_group,60 and older,6896,0.0242,0.1175,0.1970,0.9563,0.9713,False


,attribute,metric,eligible_groups,excluded_groups,lowest_group,highest_group,minimum,maximum,gap_pp
0,CODE_GENDER,false_positive_rate,2,XNA,F,M,0.089,0.144,5.515
1,CODE_GENDER,recall,2,XNA,F,M,0.357,0.449,9.219
2,CODE_GENDER,precision,2,XNA,F,M,0.232,0.260,2.734
3,CODE_GENDER,npv,2,XNA,M,F,0.932,0.949,1.700
4,CODE_GENDER,favorable_prediction_rate,2,XNA,M,F,0.825,0.892,6.720
5,CODE_GENDER,specificity,2,XNA,M,F,0.856,0.911,5.515
6,age_group,false_positive_rate,4,,60 and older,Under 30,0.024,0.220,19.597
7,age_group,recall,4,,60 and older,Under 30,0.117,0.551,43.367
8,age_group,precision,4,,60 and older,30 to under 45,0.197,0.255,5.841
9,age_group,npv,4,,Under 30,60 and older,0.932,0.956,2.467


## Three deployment questions
Main stratified evaluation targets applicants similar to the source population. Random splitting is a comparison. The nonrandom protocol trains/tunes on applicants with recorded prior Home Credit applications and tests applicants without them; history determines membership only. A higher AUC across different populations does not identify the best split.

In [21]:
def run_alternative_splits():
    torch.set_num_threads(1);df=load_data();parts=main_partitions(df)
    eligible=df.index[~df.index.isin(parts['eyeball'])]
    tr,te=train_test_split(eligible,test_size=61503,random_state=42)
    tr,va=train_test_split(tr,test_size=61002,random_state=42)
    previous=pd.read_csv('previous_application.csv',usecols=['SK_ID_CURR','SK_ID_PREV','DAYS_DECISION'])
    assert previous.SK_ID_PREV.is_unique
    known_ids=set(previous.loc[previous.DAYS_DECISION.lt(0),'SK_ID_CURR'])
    has=df.loc[eligible,'SK_ID_CURR'].isin(known_ids).to_numpy();known=eligible[has];new=eligible[~has]
    ptr,pva=train_test_split(known,test_size=.20,random_state=42,stratify=df.loc[known,'TARGET'])
    all_results=[pd.read_json('.project2_runtime/selected/metrics.json').assign(protocol='stratified')]
    for name,partition in [('random',{'train':tr,'blackbox':va,'test':te}),('no_previous_application',{'train':ptr,'blackbox':pva,'test':new})]:
        assert sum(map(len,partition.values()))==len(set().union(*map(set,partition.values())))==len(eligible)
        assert not set().union(*map(set,partition.values()))&set(parts['eyeball'])
        folder=Path('.project2_runtime/splits')/name
        _,ck,t=fit_recipe(df,partition['train'],partition['blackbox'],folder)
        # Freeze this protocol's dev-selected settings before any test inference.
        (folder/'selection.json').write_text(json.dumps({'threshold':t,'test_used_for_selection':False,'split_ids':{k:df.loc[v,'SK_ID_CURR'].tolist() for k,v in partition.items()}},indent=2))
        all_results.append(evaluate_frozen(df,partition,folder,ck,t).assign(protocol=name))
    comparison=pd.concat(all_results,ignore_index=True);comparison.to_json('.project2_runtime/split_comparison.json',orient='records',indent=2)
    return comparison


In [22]:
RERUN_SPLITS=False
saved_split_results=[{'partition': 'train', 'threshold': 0.167, 'selected_epoch': 23, 'n': 184506, 'prevalence': 0.080729082, 'auc': 0.7763498687, 'average_precision': 0.2645091805, 'accuracy': 0.859191571, 'precision': 0.2643394702, 'recall': 0.4173883854, 'f1': 0.3236840735, 'false_positive_rate': 0.1020098932, 'brier': 0.0667318255, 'log_loss': 0.2406483442, 'tn': 152309, 'fp': 17302, 'fn': 8678, 'tp': 6217, 'protocol': 'stratified'}, {'partition': 'blackbox', 'threshold': 0.167, 'selected_epoch': 23, 'n': 61002, 'prevalence': 0.0807350579, 'auc': 0.7547313325, 'average_precision': 0.2364291889, 'accuracy': 0.8525458182, 'precision': 0.2447315605, 'recall': 0.396142132, 'f1': 0.3025509808, 'false_positive_rate': 0.1073702231, 'brier': 0.0681910962, 'log_loss': 0.247404322, 'tn': 50056, 'fp': 6021, 'fn': 2974, 'tp': 1951, 'protocol': 'stratified'}, {'partition': 'eyeball', 'threshold': 0.167, 'selected_epoch': 23, 'n': 500, 'prevalence': 0.08, 'auc': 0.6865217391, 'average_precision': 0.1590306969, 'accuracy': 0.838, 'precision': 0.1846153846, 'recall': 0.3, 'f1': 0.2285714286, 'false_positive_rate': 0.1152173913, 'brier': 0.0725353956, 'log_loss': 0.2657588422, 'tn': 407, 'fp': 53, 'fn': 28, 'tp': 12, 'protocol': 'stratified'}, {'partition': 'test', 'threshold': 0.167, 'selected_epoch': 23, 'n': 61503, 'prevalence': 0.0807277694, 'auc': 0.756676284, 'average_precision': 0.2438416399, 'accuracy': 0.8551127587, 'precision': 0.2497463217, 'recall': 0.3965760322, 'f1': 0.3064829948, 'false_positive_rate': 0.1046199017, 'brier': 0.067854479, 'log_loss': 0.2463872731, 'tn': 50623, 'fp': 5915, 'fn': 2996, 'tp': 1969, 'protocol': 'stratified'}, {'partition': 'train', 'threshold': 0.166, 'selected_epoch': 29, 'n': 184506, 'prevalence': 0.0810000759, 'auc': 0.7813595719, 'average_precision': 0.2737657191, 'accuracy': 0.8566767476, 'precision': 0.2664039329, 'recall': 0.4387420542, 'f1': 0.3315132211, 'false_positive_rate': 0.1064867511, 'brier': 0.0665520504, 'log_loss': 0.2395434231, 'tn': 151505, 'fp': 18056, 'fn': 8388, 'tp': 6557, 'protocol': 'random'}, {'partition': 'blackbox', 'threshold': 0.166, 'selected_epoch': 29, 'n': 61002, 'prevalence': 0.07950559, 'auc': 0.7551412918, 'average_precision': 0.234154025, 'accuracy': 0.852578604, 'precision': 0.243531014, 'recall': 0.4055670103, 'f1': 0.3043242825, 'false_positive_rate': 0.1088117966, 'brier': 0.0672606304, 'log_loss': 0.244699955, 'tn': 50042, 'fp': 6110, 'fn': 2883, 'tp': 1967, 'protocol': 'random'}, {'partition': 'test', 'threshold': 0.166, 'selected_epoch': 29, 'n': 61503, 'prevalence': 0.0811342536, 'auc': 0.7571951115, 'average_precision': 0.240783582, 'accuracy': 0.8503975416, 'precision': 0.2443230115, 'recall': 0.4032064128, 'f1': 0.3042722117, 'false_positive_rate': 0.1101162564, 'brier': 0.0683569983, 'log_loss': 0.2476376444, 'tn': 50290, 'fp': 6223, 'fn': 2978, 'tp': 2012, 'protocol': 'random'}, {'partition': 'train', 'threshold': 0.134, 'selected_epoch': 23, 'n': 232472, 'prevalence': 0.0819281462, 'auc': 0.7744334202, 'average_precision': 0.269094705, 'accuracy': 0.8333906879, 'precision': 0.2416806634, 'recall': 0.4835135987, 'f1': 0.3222747157, 'false_positive_rate': 0.135386504, 'brier': 0.0676374882, 'log_loss': 0.2435355484, 'tn': 184531, 'fp': 28895, 'fn': 9837, 'tp': 9209, 'protocol': 'no_previous_application'}, {'partition': 'blackbox', 'threshold': 0.134, 'selected_epoch': 23, 'n': 58118, 'prevalence': 0.081919543, 'auc': 0.7557628206, 'average_precision': 0.2372916864, 'accuracy': 0.8313775422, 'precision': 0.2324519486, 'recall': 0.4597773577, 'f1': 0.3087882635, 'false_positive_rate': 0.1354648875, 'brier': 0.0689729601, 'log_loss': 0.2495719641, 'tn': 46129, 'fp': 7228, 'fn': 2572, 'tp': 2189, 'protocol': 'no_previous_application'}, {'partition': 'test', 'threshold': 0.134, 'selected_epoch': 23, 'n': 16421, 'prevalence': 0.0595578832, 'auc': 0.7637059206, 'average_precision': 0.1955018093, 'accuracy': 0.8953169722, 'precision': 0.224535316, 'recall': 0.308793456, 'f1': 0.2600086096, 'false_positive_rate': 0.0675386907, 'brier': 0.0520749241, 'log_loss': 0.1985595077, 'tn': 14400, 'fp': 1043, 'fn': 676, 'tp': 302, 'protocol': 'no_previous_application'}]
comparison=run_alternative_splits() if RERUN_SPLITS else pd.DataFrame(saved_split_results)
display(comparison[['protocol','partition','n','prevalence','auc','accuracy','precision','recall','f1','false_positive_rate','threshold']].round(4))


,protocol,partition,n,prevalence,auc,accuracy,precision,recall,f1,false_positive_rate,threshold
0,stratified,train,184506,0.0807,0.7763,0.8592,0.2643,0.4174,0.3237,0.1020,0.167
1,stratified,blackbox,61002,0.0807,0.7547,0.8525,0.2447,0.3961,0.3026,0.1074,0.167
2,stratified,eyeball,500,0.0800,0.6865,0.8380,0.1846,0.3000,0.2286,0.1152,0.167
3,stratified,test,61503,0.0807,0.7567,0.8551,0.2497,0.3966,0.3065,0.1046,0.167
4,random,train,184506,0.0810,0.7814,0.8567,0.2664,0.4387,0.3315,0.1065,0.166
5,random,blackbox,61002,0.0795,0.7551,0.8526,0.2435,0.4056,0.3043,0.1088,0.166
6,random,test,61503,0.0811,0.7572,0.8504,0.2443,0.4032,0.3043,0.1101,0.166
7,no_previous_application,train,232472,0.0819,0.7744,0.8334,0.2417,0.4835,0.3223,0.1354,0.134
8,no_previous_application,blackbox,58118,0.0819,0.7558,0.8314,0.2325,0.4598,0.3088,0.1355,0.134
9,no_previous_application,test,16421,0.0596,0.7637,0.8953,0.2245,0.3088,0.2600,0.0675,0.134


## Next step
Run `Final_Error_Analysis.ipynb` in this same directory. The submitted CSV has 81 errors from the fixed 500-person eyeball pool. Its case notes still require team verification. The Word document explains the performance, fairness, transfer limitations and hypothesis findings.